# Laboratorio de series de tiempo

Pronóstico *day-ahead* del consumo eléctrico horario de clientes portugueses. El notebook mantiene separados los bloques de entrenamiento, validación y prueba, y usa una sola implementación del backtest para todos los modelos.

**Reproducibilidad.** El entorno recomendado es Python 3.11. Las corridas costosas se activan explícitamente; las funciones reutilizables y sus pruebas viven en `src/forecasting_lab`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecasting_lab.config import ExperimentConfig
from forecasting_lab.data import (aggregate_quarter_hourly, download_uci_data,
                                  filter_complete_clients, load_raw_data,
                                  select_clients, split_panel, to_long)
from forecasting_lab.features import make_calendar_features
from forecasting_lab.dataset import PanelWindowDataset, SeriesStandardizer, count_windows
from forecasting_lab.leakage import assert_no_leakage
from forecasting_lab.backtest import backtest

CONFIG = ExperimentConfig()
EJECUTAR_CORRIDAS = False
DIRECTORIO_ARTEFACTOS = Path("artifacts")
DIRECTORIO_ARTEFACTOS.mkdir(exist_ok=True)

## 1. Datos

La fuente UCI contiene lecturas promedio en kW cada 15 minutos. Cada valor se multiplica por 0.25 h y se suman cuatro intervalos para obtener kWh horarios. El archivo tiene 96 posiciones civiles por fecha incluso durante los cambios de horario; se conservan esas etiquetas y se valida que cada hora tenga exactamente cuatro cuartos. Se retienen clientes sin faltantes y con actividad a más tardar el 1 de enero de 2012.

In [ ]:
if EJECUTAR_CORRIDAS:
    ruta_cruda = download_uci_data("data")
    crudo = load_raw_data(ruta_cruda)
    horario = aggregate_quarter_hourly(crudo)
    panel = filter_complete_clients(horario, CONFIG.data_start)
    panel = select_clients(panel, n_clients=100, seed=CONFIG.seed)
    bloques = split_panel(panel, CONFIG)
    print({"clientes": panel.shape[1], "horas": panel.shape[0],
           "observaciones_panel": panel.size})

## 2. Exploración y preparación

La exploración compara perfiles por hora y día de la semana, marca feriados nacionales y resume la escala promedio por cliente. Se usa z-score por serie con media y desviación calculadas **solo en entrenamiento**; una serie constante recibe escala 1 para evitar división por cero. Hora, día de semana y mes se codifican con seno/coseno, y feriado como indicador binario conocido a futuro.

In [ ]:
if EJECUTAR_CORRIDAS:
    covariables = make_calendar_features(panel.index)
    normalizador = SeriesStandardizer().fit(bloques.train)
    panel_normalizado = normalizador.transform(panel)
    resumen_escalas = panel.mean().agg(["min", "median", "max"])
    ventanas = {
        "train": count_windows(panel, 168, 24, CONFIG.data_start, CONFIG.train_end),
        "validation": count_windows(panel, 168, 24, CONFIG.validation_start, CONFIG.validation_end),
        "test_stride_1": count_windows(panel, 168, 24, CONFIG.test_start, CONFIG.test_end),
    }
    display(resumen_escalas, pd.Series(ventanas, name="ventanas"))

In [ ]:
if EJECUTAR_CORRIDAS:
    perfil_hora = panel.groupby(panel.index.hour).mean().mean(axis=1)
    perfil_dia = panel.groupby(panel.index.dayofweek).mean().mean(axis=1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    perfil_hora.plot(ax=axes[0], title="Perfil promedio por hora")
    perfil_dia.plot(ax=axes[1], title="Perfil promedio por día de semana")
    plt.tight_layout()

## 3. Investigación

- `nn.LSTM` y `nn.GRU`: `input_size` es el número de atributos por paso, `hidden_size` el estado, `num_layers` apila recurrentes, `batch_first=True` usa `(batch, tiempo, atributos)` y `dropout` actúa entre capas. LSTM devuelve `output`, `(h_n, c_n)`; GRU devuelve `output`, `h_n`.
- `nn.Conv1d`: con convoluciones causales dilatadas, el padding izquierdo evita usar el futuro. Para kernel `k` y dilataciones `d_i`, el campo receptivo es `1 + Σ (k-1)d_i` (por bloque; aumenta si hay varias convoluciones).
- `nn.TransformerEncoderLayer`: `d_model`, `nhead` y `dim_feedforward` fijan dimensión, cabezas y MLP interno. Como la atención no conoce orden, se suma codificación posicional antes del encoder.
- NeuralForecast y StatsForecast reciben `unique_id, ds, y`. `cross_validation` controla `n_windows`, `step_size` y `refit`; aquí el paso de prueba es 24 h.
- `BaseChronosPipeline.predict_quantiles` recibe contexto, `prediction_length=24` y niveles como `[0.1, 0.5, 0.9]`.

Un modelo local ajusta parámetros por serie; uno global comparte patrones entre clientes. *Teacher forcing* entrega al decoder el valor real anterior durante entrenamiento, por lo que debe vigilarse la brecha con inferencia. RevIN normaliza cada ventana y revierte la transformación, útil ante cambios de nivel. Pinball loss estima cuantiles: penaliza de forma asimétrica según el nivel solicitado. MASE divide MAE por el error naive estacional y permite comparar escalas; WAPE divide el error absoluto total por la demanda total y pondera más las series grandes.

## 4. Protocolo de validación y métricas

Los pesos se ajustan hasta el 30 de septiembre. La prueba contiene 92 orígenes diarios; cada origen usa historia hasta las 23:00 anteriores y produce 24 horas. MAE, RMSE, WAPE, MAPE y MASE se calculan en kWh, además de desgloses por cliente, paso, hora y semana del 22–28 de diciembre.

In [ ]:
auditoria = assert_no_leakage(
    train_end=pd.Timestamp(CONFIG.train_end),
    validation_end=pd.Timestamp(CONFIG.validation_end),
    scaler_fit_end=pd.Timestamp(CONFIG.train_end),
    early_stopping_end=pd.Timestamp(CONFIG.validation_end),
    origin=pd.Timestamp(CONFIG.test_start),
    history_end=pd.Timestamp(CONFIG.validation_end),
    future_covariate_end=pd.Timestamp(CONFIG.test_end),
    covariates_known_in_advance=True,
)
auditoria

## 5. Construcción y entrenamiento

Las clases propias (`MultiOutputMLP`, `DLinear`, `NLinear`, `LSTMForecaster`, `AutoregressiveLSTM`, `Seq2SeqForecaster`) están en `models.py`. Los adaptadores aíslan StatsForecast, LightGBM, NeuralForecast y Chronos con imports perezosos para que el pipeline base sea utilizable sin instalar todos los motores.

In [ ]:
from forecasting_lab.models import (MultiOutputMLP, DLinear, NLinear,
                                       LSTMForecaster, AutoregressiveLSTM,
                                       Seq2SeqForecaster)
from forecasting_lab.baselines import BaselineForecaster
from forecasting_lab.lightgbm_adapter import LightGBMGlobalAdapter
from forecasting_lab.chronos_adapter import ChronosBoltAdapter
from forecasting_lab.neuralforecast_adapter import NeuralForecastAdapter

# Definiciones listas para entrenamiento; no se toca prueba al seleccionar.
modelos_propios = {
    "mlp": MultiOutputMLP(context_length=168, input_size=8, horizon=24),
    "dlinear": DLinear(context_length=168, horizon=24),
    "nlinear": NLinear(context_length=168, horizon=24),
    "lstm": LSTMForecaster(input_size=8, hidden_size=64, horizon=24),
}

### 5.1 Estrategias multi-paso

Se comparan: (1) autorregresiva, que realimenta cada predicción y puede acumular error; (2) multi-salida, que proyecta simultáneamente 24 pasos; y (3) encoder-decoder, cuyo decoder recibe covariables futuras de calendario. La comparación usa MAE por paso con el mismo panel y backtest.

### 5.2 Iteraciones de hiperparámetros

Cada corrida registra configuración, `L`, `H`, meses de historia, pérdida por epoch, MAE/MASE de validación, parámetros, tiempos y memoria pico. Se exploran `L ∈ {48,168,336}` e historia `{3,12,30}` meses, además de normalización, estado oculto, capas, dropout, learning rate y batch. Solo validación decide la configuración.

In [ ]:
columnas_iteracion = [
    "modelo", "configuracion", "L", "H", "history_months",
    "epoch", "train_loss", "validation_loss", "validation_mae",
    "validation_mase", "total_parameters", "trainable_parameters",
    "epoch_seconds", "training_seconds", "inference_seconds",
    "peak_gpu_memory_mb", "seed",
]
registro_iteraciones = pd.DataFrame(columns=columnas_iteracion)

### 5.3 Modelo local contra global

La comparación fija los mismos diez clientes. El modelo local se ajusta por cliente; el global comparte parámetros y usa todas las series. Se reporta MASE por cliente para identificar dónde la transferencia entre series ayuda.

## 6. Comparación de resultados

La tabla final se ordena por MASE e incluye familia, `L`, estrategia, covariables, parámetros, tiempos, hardware, MAE, RMSE, WAPE, MASE, semana navideña y resultados probabilísticos. Las tres figuras obligatorias se regeneran desde CSV, sin copiar valores manualmente.

In [ ]:
from forecasting_lab.results import (build_comparison_table, plot_pareto,
                                       plot_horizon_mae, plot_history_sensitivity,
                                       write_predictions)

if (DIRECTORIO_ARTEFACTOS / "resultados_modelos.csv").exists():
    resultados = pd.read_csv(DIRECTORIO_ARTEFACTOS / "resultados_modelos.csv")
    print(build_comparison_table(resultados).to_string(index=False))

## 7. Artefactos reproducibles

`scripts/run_persona_a.py` registra checksum, conteos, baselines y pronósticos sin números copiados manualmente. Cada modelo escribe `unique_id, ds, modelo, y_hat`; las tablas y figuras consumen esos archivos. Las dependencias opcionales ausentes se reportan explícitamente y nunca se sustituyen por resultados inventados.

## 8. Discusión y análisis

Completar tras integrar las corridas finales: ganador frente a seasonal naive y su costo; DLinear frente a Transformers; acumulación de error multi-paso; efecto de `L`, normalización e historia; local frente a global; cercanía de Chronos zero-shot; Navidad/Año Nuevo; y desviación estándar del MASE en tres semillas. Las conclusiones deben citar exclusivamente la tabla y figuras generadas.